# Reddit Kaggle Competition: Adrian Barrios

The dataset contains 28 different emotion categories (including 'neutral'). This is a **multi-label classification** task, meaning a single comment can have multiple emotions at once.

### The Goal
Predict the **probability** (0.0 to 1.0) for each of the 28 emotion columns for every test sample.

### The Metric
Performance will be evaluated using **Macro ROC AUC**.

---

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import re
from collections import Counter
import os

# Configuration
DATA_DIR = '../data/'
BATCH_SIZE = 64
MAX_LEN = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Load the Data
Load the CSV files provided for the competition.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

# The emotion labels are all columns except ID and text
emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total emotions: {num_labels}")

train_df.head(2)

## 2. Text Preprocessing & Vocabulary
We need to convert raw text into sequences of numbers (indices) that our neural network can understand.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

# Build Vocabulary from Training Data
all_tokens = [t for text in train_df['text'] for t in tokenize(text)]
vocab_counts = Counter(all_tokens)

# TODO: Experiment with vocab size or minimum frequency
vocab = {word: i+2 for i, (word, count) in enumerate(vocab_counts.most_common(10000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)

def text_to_ids(text):
    ids = [vocab.get(w, 1) for w in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

print(f"Vocabulary size: {vocab_size}")

## 3. Dataset and DataLoader

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.texts = df['text'].apply(text_to_ids).tolist()
        self.is_test = is_test
        if not is_test:
            self.labels = df[emotion_cols].values.astype(float)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.texts[idx], dtype=torch.long)
        if self.is_test:
            return x
        y = torch.tensor(self.labels[idx], dtype=torch.float)
        return x, y

# Split for internal validation
train_split, val_split = train_test_split(train_df, test_size=0.1, random_state=42)

train_loader = DataLoader(EmotionDataset(train_split), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_split),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(EmotionDataset(test_df, is_test=True), batch_size=BATCH_SIZE)

## 4. The Model: Simple RNN Baseline
This is a very basic model. Your goal is to improve this architecture.

In [ ]:
class SimpleRNNBaseline(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        # TODO: Hint - Try using pre-trained GloVe or FastText embeddings instead of learning from scratch
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # TODO: Hint - RNNs are simple. Try using LSTM or GRU with bidirectionality
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        # TODO: Hint - Instead of just the last hidden state, try Global Max Pooling or Attention
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        # We take the last hidden state
        return self.fc(hidden.squeeze(0))

model = SimpleRNNBaseline(vocab_size, 100, 128, num_labels).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()  # Standard for multi-label tasks

## 5. Training and Validation
We train the model and monitor the **Validation ROC AUC** to detect overfitting.

In [ ]:
EPOCHS = 3  # Start small, train more epochs in reality

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            logits = model(x.to(DEVICE))
            val_preds.append(torch.sigmoid(logits).cpu().numpy())
            val_targets.append(y.numpy())

    auc = roc_auc_score(np.vstack(val_targets), np.vstack(val_preds), average='macro')
    print(f"Epoch {epoch+1} | Val ROC AUC: {auc:.4f}")  # TODO: Monitor train AUC as well to detect overfitting/underfitting

    torch.save(model.state_dict(), f'model_epoch_{epoch+1}.pth')
    print(f"Model saved to model_epoch_{epoch+1}.pth")  # TODO: Upload your best model to Google Drive to use in the inference notebook

## 6. Generate Submission
Finally, we run the model on the test set and save the probabilities.

In [ ]:
model.eval()
probs = []
with torch.no_grad():
    for x in test_loader:
        logits = model(x.to(DEVICE))
        probs.append(torch.sigmoid(logits).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")

---
### Ideas to Beat the Baseline:
1. **Modern Architecture**: Upgrade the RNN to a **Bi-LSTM** or **Bi-GRU**.
2. **Better Features**: Use **GloVe** pre-trained word vectors (they are available in the `data/glove` folder!).
3. **Attention Mechanism**: Add an Attention layer so the model can focus on the most important words in each sentence.
4. **Regularization**: Add **Dropout** to prevent the model from memorizing the training text.
5. **Data Augmentation**: Try randomly swapping or deleting words during training to make your model more robust.
6. **Transformer Fine-Tuning**: If you're feeling adventurous, try fine-tuning a pre-trained model like **BERT** or **DistilBERT**.